# 模組一｜SAM 3 航照分割（Colab）

把國土測繪中心正射影像切成小圖，用 SAM 3 文字提示分割出人行道、樹冠、灌木、車道、行穿線，輸出與原圖同座標的遮罩 GeoTIFF。

**執行前準備**
1. Runtime → Change runtime type → **T4 GPU**
2. 已在 Hugging Face 取得 [facebook/sam3](https://huggingface.co/facebook/sam3) 存取權限，並建立 Access Token（Read）
3. 把本機的 `data/processed/<id>_ortho_z19.tif` 上傳到 Google Drive 的 `taoyuan-corridor/` 資料夾

**輸出**：`taoyuan-corridor/<id>_mask_<類別>.tif`（下載回本機的 `data/processed/`，再執行 `corridor_metrics.py`）

In [ ]:
# ==== 參數（可調整）====
SITE_ID = "taoyuan_hospital"
DRIVE_DIR = "/content/drive/MyDrive/taoyuan-corridor"
PROMPTS = ["sidewalk", "tree canopy", "shrub", "road", "crosswalk"]
SCORE_THRESHOLD = 0.4   # 低於此分數的實例不採用；人行道漏抓多就調低、誤抓多就調高
CHIP = 1008             # 小圖邊長（px）
OVERLAP = 128           # 小圖重疊（px）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 安裝 SAM 3 與地理套件（若 torch 版本不相容，依 SAM 3 README 安裝指定版本後重啟執行階段）
!pip -q install rasterio
!git clone -q https://github.com/facebookresearch/sam3.git /content/sam3 || true
!pip -q install -e /content/sam3
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
# 登入 Hugging Face（貼上 Read 權限的 Access Token）
from huggingface_hub import login
login()

In [ ]:
import numpy as np, rasterio
from PIL import Image
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

model = build_sam3_image_model()
processor = Sam3Processor(model)

src_path = f"{DRIVE_DIR}/{SITE_ID}_ortho_z19.tif"
with rasterio.open(src_path) as src:
    rgb = src.read().transpose(1, 2, 0)
    profile = src.profile
H, W = rgb.shape[:2]
print("影像大小", W, "x", H)

In [ ]:
def to_mask(m, shape):
    m = m.detach().float().cpu().numpy() if hasattr(m, "detach") else np.asarray(m, dtype=np.float32)
    m = np.squeeze(m)
    if m.shape != shape:  # 保險：尺寸不同時縮放回小圖大小
        m = np.asarray(Image.fromarray((m > 0.5).astype(np.uint8) * 255).resize(shape[::-1], Image.NEAREST)) / 255
    return m > 0.5

def chip_origins(n, chip, overlap):
    step = chip - overlap
    xs = list(range(0, max(n - chip, 0) + 1, step))
    if xs[-1] + chip < n:
        xs.append(n - chip)
    return xs

masks = {p: np.zeros((H, W), dtype=bool) for p in PROMPTS}
ys, xs = chip_origins(H, CHIP, OVERLAP), chip_origins(W, CHIP, OVERLAP)
total = len(ys) * len(xs)
done = 0
with torch.inference_mode():
    for y0 in ys:
        for x0 in xs:
            tile = rgb[y0:y0 + CHIP, x0:x0 + CHIP]
            state = processor.set_image(Image.fromarray(tile))
            for p in PROMPTS:
                out = processor.set_text_prompt(state=state, prompt=p)
                scores = out["scores"]
                scores = scores.detach().cpu().numpy() if hasattr(scores, "detach") else np.asarray(scores)
                for m, s in zip(out["masks"], scores):
                    if float(s) >= SCORE_THRESHOLD:
                        masks[p][y0:y0 + CHIP, x0:x0 + CHIP] |= to_mask(m, tile.shape[:2])
            done += 1
            print(f"{done}/{total}", end="\r")
for p in PROMPTS:
    print(f"{p:12s} 覆蓋 {masks[p].mean():.1%}")

In [ ]:
# 寫出遮罩 GeoTIFF（與原圖同座標）
prof = profile.copy()
prof.update(count=1, dtype="uint8", compress="deflate", photometric="minisblack", nodata=None)
prof.pop("jpeg_quality", None)
for p in PROMPTS:
    out = f"{DRIVE_DIR}/{SITE_ID}_mask_{p.replace(' ', '_')}.tif"
    with rasterio.open(out, "w", **prof) as dst:
        dst.write(masks[p].astype(np.uint8), 1)
    print("→", out)

In [ ]:
# 目視檢查：人行道（藍）、樹冠＋灌木（綠）疊在航照上，取中央 1500px
import matplotlib.pyplot as plt
cy, cx, r = H // 2, W // 2, 750
view = rgb[cy - r:cy + r, cx - r:cx + r].astype(float)
sw = masks["sidewalk"][cy - r:cy + r, cx - r:cx + r]
cv = (masks["tree canopy"] | masks["shrub"])[cy - r:cy + r, cx - r:cx + r]
view[sw] = view[sw] * 0.4 + np.array([30, 120, 255]) * 0.6
view[cv] = view[cv] * 0.5 + np.array([0, 220, 60]) * 0.5
plt.figure(figsize=(12, 12)); plt.imshow(view.clip(0, 255).astype(np.uint8)); plt.axis("off")